### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="asp_potassco_classification",
    dataset_year="2014",
    domain_str="technology & internet",
    # Data Source
    dataset_source="ASlib",
    original_dataset_source_download_link="https://github.com/coseal/aslib_data/tree/master/ASP-POTASSCO",
    download_description=r"""
wget https://raw.githubusercontent.com/coseal/aslib_data/refs/heads/master/ASP-POTASSCO/algorithm_runs.arff \
&& wget https://raw.githubusercontent.com/coseal/aslib_data/refs/heads/master/ASP-POTASSCO/feature_values.arff \
&& mkdir -p local-data-warehouse/asp_potassco_classification \
&& mv feature_values.arff algorithm_runs.arff local-data-warehouse/asp_potassco_classification/
""",
    # References
    academic_reference_bibtex="""@article{hoos2014claspfolio,
  title={claspfolio 2: Advances in algorithm selection for answer set programming},
  author={Hoos, Holger and Lindauer, Marius and Schaub, Torsten},
  journal={Theory and Practice of Logic Programming},
  volume={14},
  number={4-5},
  pages={569--585},
  year={2014},
  publisher={Cambridge University Press}
}
@article{bischl_aslib_2016,
	title = {{ASlib}: {A} {Benchmark} {Library} for {Algorithm} {Selection}},
	number = {237},
	journal = {Artificial Intelligence Journal (AIJ)},
	author = {Bischl, Bernd and Kerschke, Pascal and Kotthoff, Lars and Lindauer, Marius and Malitsky, Yuri and Fréchette, Alexandre and Hoos, Holger H. and Hutter, Frank and Leyton-Brown, Kevin and Tierney, Kevin and Vanschoren, Joaquin},
	year = {2016},
	pages = {41--58}
}
""",
    academic_reference_bibtex_key="hoos2014claspfolio,bischl_aslib_2016",
    license="GPLv3",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We get the data from ASlib and merge them into one file.

- We treat it as a multiclass classification task to solve the algorithm selection task as in the OpenML version (https://openml.org/d/41705).
- We drop all cases where no algorithm was able to finish before the timeout as these are essentially random labels.
- We resole the instance ID to a task ID by mapping the instance ID back to the source task based on their naming convention. From our understanding, the data contains multiple instance from the same solver task, differing only in seed or task configurations. We want to avoid having samples from the same task in both train and test set, as this would leak what algorithms are best for this task, and since in real-world you might not have samples from your new task. Thus, we treat the data as a grouped task, where we aim to generalize the predicting the best algorithm to new tasks across a set of instances of this task.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="algorithm",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss", # or RMSE on runtime, or PAR10?
    # For classification
    stratify_on="algorithm",
    # For grouped data
    group_on="task_id",
    group_labels="per_sample",
)

## Preprocessing

In [ ]:
import arff
import pandas as pd
import uuid
import numpy as np


def load_arff(path) -> pd.DataFrame:
    with open(path, encoding="utf-8") as f:
        data = arff.load(f)
    df = pd.DataFrame(data["data"], columns=[a[0] for a in data["attributes"]])
    return df


df_features = load_arff(dataset_mold.path / "feature_values.arff").drop(columns=["repetition"])
df_algo_runs = load_arff(dataset_mold.path / "algorithm_runs.arff")

# Only keep instances for which we have meaningful labels
df_algo_runs = df_algo_runs[df_algo_runs["runstatus"] == "ok"]
# Reduce to best algorithm per instance
df_algo_runs = df_algo_runs.loc[
    df_algo_runs.groupby('instance_id')['runtime'].idxmin(),
    ['instance_id', 'algorithm']
].reset_index(drop=True)

# Merge
df = df_algo_runs.merge(df_features, on="instance_id", how="left")
print("Merged data shape:", df.shape)


# Map instance ID to group ID
df["task_id"]  = df["instance_id"].str.rsplit("/", n=1).str[0]
# map it to a uuid
mapping = {val: uuid.uuid4().hex[:12] for val in df["task_id"].unique()}
df["task_id"] = df["task_id"].map(mapping)
df = df.drop(columns=[
    "instance_id",
    # Duplicated columns
    "Frac_Removed_Nogood-1",
    "Frac_Removed_Nogood-2",
])

as_cat_type = ["task_id", "algorithm"]
df[as_cat_type] = df[as_cat_type].astype("category")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For Grouped Non-IID data
split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
    random_state=split_random_state,
)


splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()